<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Fundamentals of Generative AI</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 2a: Amazon Bedrock API Walkthrough</h2>
</div>

Amazon Bedrock is a fully managed service that makes foundation models (FMs) from Amazon and third-party model providers easily accessible through an API. This notebook covers the Amazon Bedrock API using SDK for Python (Boto3).

In this lab, we will cover topics such as:
- Setting up and accessing the Bedrock service using Boto3
- Exploring available Large Language Models (LLMs) in Bedrock
- Performing Bedrock API calls with various customization options
- Understanding and manipulating model parameters for text generation

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents -->
<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2874A6; border-bottom: 1px solid #2874A6; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2E86C1; font-weight: bold; text-decoration: none;">1. Installation and API calls</a></p>
    <p><a href="#section2" style="color: #2E86C1; font-weight: bold; text-decoration: none;">2. Accessing the Bedrock service using the SDK for Python</a></p>
    <p><a href="#section3" style="color: #2E86C1; font-weight: bold; text-decoration: none;">3. Bedrock API Call</a></p>
    <p><a href="#section4" style="color: #2E86C1; font-weight: bold; text-decoration: none;">4. Quizzes</a></p>
</div>

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">1. Installation and API calls</h2>
</div>
(<a href="#0">Go to top</a>)

We first install recent versions of boto3 and botocore (defined in a file called `requirements.txt`).

In [ ]:
!pip install --no-deps -q -r ../requirements.txt

We can access the Bedrock service through boto3 by providing the service name, region name and endpoint URL.

<!-- Section Header -->
<div id="#section2" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">2. Accessing the Bedrock service using the SDK for Python</h2>
</div>
(<a href="#0">Go to top</a>)

In [ ]:
import json, boto3

session = boto3.session.Session()

bedrock = session.client(
    service_name="bedrock",
    region_name=session.region_name,
)

Let's take a look at the available Large Language Models (LLMs) in Bedrock.

In [ ]:
bedrock.list_foundation_models()["modelSummaries"]
# uncomment the command below to display only model IDs instead
# [m["modelId"] for m in bedrock.list_foundation_models()["modelSummaries"][0:]]

We use the service name 'bedrock-runtime' for inference. 

In [ ]:
# For inference
bedrock_inference = session.client(
    service_name="bedrock-runtime",
)

Model IDs can be used to select a specific model in the API calls. 

This demo is focusing on the __Nova Micro__ model from __Amazon__. We will use the model with id: `amazon.nova-micro-v1:0`

---

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">3. Bedrock API Call</h2>
</div>
(<a href="#0">Go to top</a>)

Bedrock API has the following parameters:
* __`body`:__ The message body that includes the input text and model parameters. Model parameters help customize the models and the generated outputs.
* __`modelId`:__ Identifier of the model. We can pick a model from the list of models printed in the previous code block.
* __`accept`:__ The desired type of the inference body in the response.
* __`contentType`:__ The type of the input data in the request body.

<br>

__Model parameters:__

These parameters are provided within the body of the API call. Basically, we can control the randomness and the length of the generated sequences. 

__Randomness:__ 

* __`temperature`:__ Controls the randomness of the generated sequences. This parameter is between zero and one. When set closer to zero, the model tends to select higher probability words. When set further away from zero, the model may select lower-probability words. Temperature of zero gives the same output for the same input at every run.
* __`topP`:__ Top P defines a cut-off based on the sum of probabilities of the potential choices. With this cut-off, the model only selects from the most probable words whose probabilities sum up to the topP value. 

__Length:__ 

* __`maxTokens`:__ Controls the maximum number of tokens in the generated response.
* __`stopSequences`:__ A sequence of characters to stop the model from generating its output.

Let's define a function that will build our prompt as well as pass some model parameters to Bedrock. Once the call parameters are ready, we can use the __converse()__ function to send the data and collect the response from the model. Then, we return the response at the end.

**Note:** Different models may have different model parameters. Please refer to the [documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/service_code_examples_bedrock-runtime.html) for more details.

In [ ]:
def send_prompt(prompt_data, temperature=0.0, top_p=1.0, max_token_count=1000):

    # select model
    model_id = "amazon.nova-micro-v1:0"
    
    # Build the message to call Converse API
    message_content = []
    message_content.append({"text": prompt_data})

    messages = [
            {
                "role": "user",
                "content": message_content,
            }
      ]
    inf_params = {"maxTokens": max_token_count, "topP": top_p, "temperature": temperature}
    response = bedrock_inference.converse(
            modelId=model_id, messages=messages, inferenceConfig=inf_params
    )
    # Process and return the response

    return response["output"]["message"]["content"][0]["text"]

Let's construct a simple API call and run it. 

As a sample text input, we use the following: __"Can you name a few real-life applications of natural language processing?"__. 

For inference parameters, we set the __`temperature`__ to 0.

In [ ]:
from IPython.display import Markdown, display

prompt_data = (
    """Can you name a few real-life applications of natural language processing?"""
)

display(Markdown(prompt_data))
display(Markdown(send_prompt(prompt_data, temperature=0.0)))

This provides a list of real-life NLP applications. 

If we want to generate slightly different looking outputs, we can increase the __`temperature`__ value. Let's also set the __`maxTokens`__ parameter this time.

Changing the temperature parameter will create a slightly different looking list.

In [ ]:
prompt_data = (
    """Can you name a few real-life applications of natural language processing?"""
)

display(Markdown(prompt_data))
display(Markdown(send_prompt(prompt_data, temperature=0.5, max_token_count=450)))

In addition to the __`temperature`__ and __`maxTokens`__ parameters, we can also add in the __`topP`__ parameter to set a cut-off for the sum of the probabilities of the potential words.

In [ ]:
prompt_data = (
    """Can you name a few real-life applications of natural language processing?"""
)

display(Markdown(prompt_data))
display(
    Markdown(send_prompt(prompt_data, max_token_count=450, temperature=0.5, top_p=0.7))
)

<!-- Section Header -->
<div id="section4" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">4. Quiz Questions</h2>
</div>
(<a href="#0">Go to top</a>)

Well done on completing the lab! Now, it's time for a brief knowledge assessment.

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/challenge.png" alt="Challenge" width="100" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Try it Yourself!</h4>
        <p>Answer the following questions to test your understanding of using LLMs for inference.</p>
    </div>
</div>

In [ ]:
import sys
sys.path.append('..')
from mlu_utils.quiz_questions import *

lab2a_question1.display()

# Thank you!

<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>